In [1]:
from pathlib import Path
import torch
import torch.nn as nn
import os
import sys
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torchinfo import summary

In [2]:
ROOT_DIR= Path.cwd().parent
ROOT_DIR

WindowsPath('D:/Code Module 6/Excercise_Week_2_Module_6')

In [3]:
DATA_DIR = ROOT_DIR/'data/'
BASE_MODEL_DIR = ROOT_DIR/'model_base/'
SRC_DIR = ROOT_DIR/'src/'

In [4]:
# Thêm các thư mục vào không gian tìm kiếm của notebooks
sys.path.extend([str(ROOT_DIR), str(DATA_DIR), str(BASE_MODEL_DIR), str(SRC_DIR)])

In [5]:
from ResNet_model import *
from precessing_data import *

# Chuẩn bị dữ liệu test và tiền xử lý dữ liệu cho TestSet

In [6]:
TEST_PATH = str(DATA_DIR/'scenes_classification'/'val/')

In [7]:
label2idex, idx2label = Code_decode_label(TEST_PATH)

In [8]:
label2idex

{'buildings': 0,
 'forest': 1,
 'glacier': 2,
 'mountain': 3,
 'sea': 4,
 'street': 5}

In [9]:
num_classes = len(label2idex)

In [10]:
mean = torch.tensor([0.4306, 0.4573, 0.4538])
std = torch.tensor([0.2603, 0.2590, 0.2901])

In [11]:
img_paths, labels = get_imgPath_label(TEST_PATH, label2idex)
img_paths

['D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20057.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20060.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20061.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20064.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20073.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20074.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20078.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20083.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\scenes_classification\\val\\buildings\\20094.jpg',
 'D:\\Code Module 6\\Excercise_Week_2_Module_6\\data\\s

In [12]:
test_dataset = ImageDataSet(img_paths= img_paths, labels=labels,transforms=[transforms.Resize((224,224)),transforms.Normalize(mean=mean, std=std)])

In [13]:
test_loader = DataLoader(test_dataset,batch_size=128)

# khởi tạo mô hình và load bộ trọng số đã huấn luyện

In [14]:
classifier = ResNet18(num_classes=num_classes)

In [15]:
summary(classifier.model, input_size=(128,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
Sequential                               [128, 6]                  --
├─Conv2d: 1-1                            [128, 64, 112, 112]       9,472
├─BatchNorm2d: 1-2                       [128, 64, 112, 112]       128
├─ReLU: 1-3                              [128, 64, 112, 112]       --
├─MaxPool2d: 1-4                         [128, 64, 56, 56]         --
├─ResidualBlock: 1-5                     [128, 64, 56, 56]         --
│    └─Sequential: 2-1                   [128, 64, 56, 56]         --
│    │    └─Conv2d: 3-1                  [128, 64, 56, 56]         36,928
│    │    └─BatchNorm2d: 3-2             [128, 64, 56, 56]         128
│    │    └─ReLU: 3-3                    [128, 64, 56, 56]         --
│    │    └─Conv2d: 3-4                  [128, 64, 56, 56]         36,928
│    │    └─BatchNorm2d: 3-5             [128, 64, 56, 56]         128
├─ReLU: 1-6                              [128, 64, 56, 56]         --
├

In [16]:
weigth_path = str(ROOT_DIR/'model_weight.pth')

In [17]:
classifier.model.load_state_dict(torch.load(weigth_path))

<All keys matched successfully>

In [18]:
with torch.no_grad():
    classifier.model.eval()
    total_acc = 0
    num_sample = len(test_dataset)
    for X_test, y_test in test_loader:
        num_batch_test = X_test.shape[0]
        X_test, y_test = X_test.to(classifier.device), y_test.to(classifier.device)
        logits = classifier.forward(X_test)
        total_acc += classifier.get_accuracy(logits, y_test)*num_batch_test
    print(f"Độ chính xác của mô hình ResNet18 trên tập test = {(total_acc/num_sample):.4f}")

Độ chính xác của mô hình ResNet18 trên tập test = 0.8800
